In [27]:
path="../2-RAG/NovaS.pdf"

In [28]:
import os


if os.path.exists(path):
    print("path is correct")
else:
    print("not active")



path is correct


In [29]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
#from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings


print("Imports successful!")

Imports successful!


In [30]:
from dotenv import load_dotenv
load_dotenv()


if os.environ.get("OPENAI_API_KEY"):
    print("pass")
else:
    print("Not set")

pass


step 1 : load data from pdf and convert into text:


In [31]:
doc=PyPDFLoader(path).load()
full_text="\n".join([p.page_content for p in doc])
full_text

'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting solutions to make better business decisions. \nDuring the first year of operations, the company worked mostly with local startups that did \nnot have large 

In [32]:
full_text

'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting solutions to make better business decisions. \nDuring the first year of operations, the company worked mostly with local startups that did \nnot have large 


#step 2, creating the chunks of text_data

In [33]:
chunker=SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=60
    )

semantic_chunks=chunker.create_documents([full_text])
semantic_chunks

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='At the \nbeginning, the company did not have large investments or a big office.'),
 Document(metadata={}, page_content='Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.'),
 Document(metadata={}, page_content='Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting 

In [ ]:
semantic_chunks[0]
#now it create the similerity on the base on percential from previous text and next text if percentil is greater 60
#this is power of semntic chunk , it will not create the chunk on the basic on sentense

Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.')

In [35]:
len(semantic_chunks)

28

step 3-b , vector database automatically create the embedding so now creating embedding and store them into vector databse
use croma db,
we can use in memory and persist data so that we can use it later


In [37]:
from langchain_community.vectorstores import Chroma


In [38]:
embed_model=OpenAIEmbeddings(model="text-embedding-3-small")
embed_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x115308910>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x1153091d0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [40]:
path_db="/Users/bilal/Downloads/AI/4-Tool Binding/chroma_db_semantic"

In [41]:
c_db=Chroma.from_documents(semantic_chunks,embed_model,persist_directory=path_db)

In [42]:
c_db

#step 4:now if i use db next day so create the connection

In [43]:
chroma_db_connection=Chroma(persist_directory=path_db,embedding_function=embed_model)
chroma_db_connection

/var/folders/2z/pqpq9wcs0dv4jfbgkp96j3780000gn/T/ipykernel_11426/1918665043.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_connection=Chroma(persist_directory=path_db,embedding_function=embed_model)


In [45]:
chroma_db_connection.similarity_search("what is novas",k=3) #k is top 2 chunks, most relevant

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='The founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make bette

In [46]:
answer=chroma_db_connection.similarity_search("By 2019, how many employees were there?",k=3) #k is top 2 chunks, most relevant

In [47]:
answer

[Document(metadata={}, page_content='The \nnumber of employees increased again, and the company also started hiring people who \nhad experience in big data technologies. This helped the organization build stronger \ntechnical capabilities and handle larger clients. Another important milestone came in 2022 when the organization decided to invest more \ntime in research and learning new technologies. Instead of working only on client projects, \nthe company created a small internal research group.'),
 Document(metadata={}, page_content='The organization began working with \ncompanies from different industries such as retail, education, and small healthcare \nproviders. Each industry had different challenges, and this helped the team improve its \nunderstanding of real-world business problems. The company also started focusing more \non improving the quality of its work instead of simply increasing the number of projects. The founders believed that long-term growth would come only if the 

In [48]:
for a in answer:
    reply=a.page_content

print(reply)

The number of employees 
increased to more than fifty, and the company started working with larger clients from 
different industries.


In [50]:
answer=chroma_db_connection.similarity_search("when was NovaSphere organization founded?",k=3) #k is top 2 chunks, most relevant

In [51]:
answer

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='The founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make bette